# 17 — DPO Preference Training

After supervised fine-tuning (NB16), this notebook runs **Direct Preference
Optimization (DPO)** to teach the model which outputs are correct vs incorrect.

## Data Sources

1. **NB14 rejects** — `data/04_validated/rejects_for_dpo.jsonl` — samples that
   looked like ARO but failed `aro check`.
2. **NB19 round rejects** — `data/rounds/round_N_rejects.jsonl` — samples that
   failed during iterative generation.
3. **Fresh pairs** — we generate N candidates per prompt with the SFT model,
   then use `aro check` to label chosen/rejected.

## Approach

DPO needs triples: `(prompt, chosen_response, rejected_response)`.  We build
these by pairing valid samples with nearby invalid ones for the same prompt.
Rejected samples are filtered to keep only "close but wrong" — they must look
like ARO (contain `(` and `<`) but fail syntax validation.

**Input:**  SFT model from NB16 + reject files from NB14/NB19

**Output:** `../models/dpo/adapter/` — DPO-trained adapter
            `../models/dpo/fused/`   — merged model ready for packaging

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'mlx-lm'], check=False)

In [ ]:
import json
import os
import re
import random
import subprocess
import sys
import tempfile
import importlib
from pathlib import Path
from collections import Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))
import config; importlib.reload(config); from config import *

# ── Paths ──────────────────────────────────────────────────────────────────────
DPO_DATA_DIR   = DATA_ROOT / 'dpo'
DPO_DATA_DIR.mkdir(parents=True, exist_ok=True)

DPO_MODEL_DIR  = MODELS_DIR / 'dpo'
DPO_ADAPTER    = DPO_MODEL_DIR / 'adapter'
DPO_FUSED      = DPO_MODEL_DIR / 'fused'
DPO_ADAPTER.mkdir(parents=True, exist_ok=True)

# ── Find the best SFT model to start from ────────────────────────────────────
def _find_latest_fused():
    """Find the latest fused SFT model (from NB16 or NB19)."""
    for parent in [MODELS_DIR / 'iterative', FINETUNE_MODELS_DIR]:
        if not parent.exists():
            continue
        rounds = sorted(parent.glob('round_*/fused'), reverse=True)
        for r in rounds:
            if (r / 'config.json').exists():
                return str(r)
    # Fall back to base model
    return MODEL_ID

SFT_MODEL = _find_latest_fused()
print(f'SFT model:    {SFT_MODEL}')
print(f'DPO adapter:  {DPO_ADAPTER}')
print(f'DPO fused:    {DPO_FUSED}')

## 1. Collect rejected samples

In [ ]:
def load_rejects():
    """Load all rejected samples from NB14 and NB19."""
    rejects = []

    # NB14 rejects
    nb13_rejects = DATA_ROOT / '04_validated' / 'rejects_for_dpo.jsonl'
    if nb13_rejects.exists():
        with open(nb13_rejects) as f:
            for line in f:
                if line.strip():
                    rejects.append(json.loads(line))
        print(f'  NB14 rejects: {len(rejects)}')

    # NB19 round rejects
    rounds_dir = DATA_ROOT / 'rounds'
    if rounds_dir.exists():
        n_round_rejects = 0
        for rfile in sorted(rounds_dir.glob('round_*_rejects.jsonl')):
            with open(rfile) as f:
                for line in f:
                    if line.strip():
                        rejects.append(json.loads(line))
                        n_round_rejects += 1
        if n_round_rejects:
            print(f'  NB19 round rejects: {n_round_rejects}')

    print(f'  Total rejects loaded: {len(rejects)}')
    return rejects


def is_close_but_wrong(output):
    """Filter: keep only rejects that look like ARO but have syntax errors.
    Drop empty, all-comments, or completely off-topic outputs."""
    if not output or len(output.strip()) < 30:
        return False
    # Must contain at least one feature set opening and one variable reference
    has_paren = '(' in output
    has_angle = '<' in output
    if not (has_paren and has_angle):
        return False
    # Reject if >70% comment lines
    lines = [l.strip() for l in output.splitlines() if l.strip()]
    if not lines:
        return False
    comment_lines = sum(1 for l in lines if l.startswith('(*'))
    if comment_lines / len(lines) > 0.7:
        return False
    return True


all_rejects = load_rejects()
filtered_rejects = [r for r in all_rejects if is_close_but_wrong(r.get('output', ''))]
print(f'After close-but-wrong filter: {len(filtered_rejects)} rejects')

## 2. Load valid samples (chosen responses)

In [ ]:
def load_valid_samples():
    """Load validated code_generation samples as 'chosen' responses."""
    valid = []
    valid_file = DATA_ROOT / '04_validated' / 'valid_samples.jsonl'
    if valid_file.exists():
        with open(valid_file) as f:
            for line in f:
                if line.strip():
                    s = json.loads(line)
                    if s.get('task_type') in {'code_generation', 'translation', 'debugging'}:
                        valid.append(s)
    # Also load from training set
    train_file = DATA_ROOT / '05_dataset' / 'train.jsonl'
    if train_file.exists():
        with open(train_file) as f:
            for line in f:
                if line.strip():
                    s = json.loads(line)
                    if s.get('task_type') == 'code_generation':
                        # Extract output from messages format
                        msgs = s.get('messages', [])
                        asst = [m['content'] for m in msgs if m['role'] == 'assistant']
                        user = [m['content'] for m in msgs if m['role'] == 'user']
                        if asst and user:
                            valid.append({
                                'instruction': user[0],
                                'output': asst[0],
                                'task_type': 'code_generation',
                            })
    print(f'Valid (chosen) samples: {len(valid)}')
    return valid

valid_samples = load_valid_samples()

## 3. Build DPO pairs

In [ ]:
def build_dpo_pairs(valid_samples, rejected_samples):
    """Build (prompt, chosen, rejected) triples for DPO training.

    Strategy:
    1. If a reject has the same instruction as a valid sample, pair them directly.
    2. Otherwise, pair each reject with a random valid sample's output
       (using the reject's instruction as the prompt). The chosen response
       shows what good ARO looks like for a similar task.
    """
    pairs = []

    # Index valid by instruction prefix for matching
    valid_by_prefix = {}
    for v in valid_samples:
        instr = v.get('instruction', '')[:100]
        if instr:
            valid_by_prefix[instr] = v

    # Build system prompt
    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)

    matched, unmatched = 0, 0
    for rej in rejected_samples:
        rej_instr = rej.get('instruction', '')
        rej_output = rej.get('output', '')
        if not rej_instr or not rej_output:
            continue

        # Try exact prefix match
        prefix = rej_instr[:100]
        chosen_sample = valid_by_prefix.get(prefix)

        if chosen_sample:
            chosen_output = chosen_sample['output']
            matched += 1
        elif valid_samples:
            # Pick a random valid sample as the chosen response
            chosen_output = random.choice(valid_samples)['output']
            unmatched += 1
        else:
            continue

        # Skip if chosen and rejected are identical
        if chosen_output.strip() == rej_output.strip():
            continue

        pairs.append({
            'prompt': [
                {'role': 'system', 'content': sys_prompt},
                {'role': 'user', 'content': rej_instr},
            ],
            'chosen': [{'role': 'assistant', 'content': chosen_output}],
            'rejected': [{'role': 'assistant', 'content': rej_output}],
        })

    print(f'DPO pairs built: {len(pairs)}')
    print(f'  Matched by instruction: {matched}')
    print(f'  Random pairing:         {unmatched}')
    return pairs

dpo_pairs = build_dpo_pairs(valid_samples, filtered_rejects)

## 4. Generate fresh DPO pairs from SFT model

In [ ]:
def aro_check(code):
    """Run aro check on code. Returns True if valid, False otherwise."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp)
            (p / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', str(p)],
                               capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return False


def extract_aro_code(text):
    """Extract ARO code from markdown-fenced or raw text."""
    blocks = re.findall(r'```aro\n(.*?)```', text, re.DOTALL)
    if blocks:
        return '\n'.join(b.strip() for b in blocks)
    # No fences — try raw
    return text.strip()


def generate_fresh_dpo_pairs(model, tokenizer, generate_fn, n_prompts=50, n_candidates=4):
    """Generate N candidates per prompt, use aro check to label chosen/rejected."""
    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)
    fresh_pairs = []

    # Use a variety of prompts
    prompts = [
        'Write an ARO feature set that retrieves a user by ID and returns an OK response.',
        'Write an ARO Application-Start that starts an HTTP server and keeps it alive.',
        'Write an ARO event handler that processes OrderCreated events and logs the order.',
        'Write an ARO feature set that reads a JSON file and iterates over its items.',
        'Write an ARO API endpoint that creates a new item in a repository.',
        'Write an ARO feature set that fetches data from an external URL and transforms it.',
        'Write an ARO feature set that filters a list of items by price > 100.',
        'Write an ARO Application-Start that watches a directory for file changes.',
        'Write an ARO feature set that computes the hash of a password and stores it.',
        'Write an ARO feature set that splits a CSV line and logs each field.',
        'Write an ARO WebSocket handler that broadcasts messages to all connections.',
        'Write an ARO feature set that uses a state machine to process orders.',
        'Write an ARO feature set that merges two collections and returns the union.',
        'Write an ARO feature set that renders an HTML template with user data.',
        'Write an ARO feature set that schedules a task to run every 5 minutes.',
        'Write an ARO feature set that extracts query parameters from a request.',
        'Write an ARO feature set that uses date arithmetic to compute a deadline.',
        'Write an ARO feature set that validates an email format and returns errors.',
        'Write an ARO feature set that copies a file from source to destination.',
        'Write an ARO feature set that emits a custom event with metadata.',
    ]

    # Extend with random variations
    domains = ['blog', 'inventory', 'payment', 'notification', 'report',
               'search', 'analytics', 'auth', 'chat', 'monitoring']
    for d in domains:
        prompts.append(f'Write a complete ARO HTTP API for a {d} service with CRUD operations.')

    random.shuffle(prompts)
    prompts = prompts[:n_prompts]

    for i, prompt_text in enumerate(prompts):
        messages = [
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user', 'content': prompt_text},
        ]
        chat_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        chosen_list, rejected_list = [], []

        for _ in range(n_candidates):
            try:
                response = generate_fn(
                    model, tokenizer, prompt=chat_prompt,
                    max_tokens=1024, verbose=False,
                    temp=0.7,  # higher temp for diversity
                )
                code = extract_aro_code(response)
                if aro_check(code):
                    chosen_list.append(response)
                elif is_close_but_wrong(response):
                    rejected_list.append(response)
            except Exception as e:
                print(f'  Generation error: {e}')
                continue

        # Build pairs from this prompt's candidates
        if chosen_list and rejected_list:
            for chosen in chosen_list:
                for rejected in rejected_list:
                    fresh_pairs.append({
                        'prompt': messages,
                        'chosen': [{'role': 'assistant', 'content': chosen}],
                        'rejected': [{'role': 'assistant', 'content': rejected}],
                    })

        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(prompts)} prompts processed, {len(fresh_pairs)} pairs so far')

    print(f'Fresh DPO pairs generated: {len(fresh_pairs)}')
    return fresh_pairs


# Load model and generate
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()
print(f'Loading SFT model from {SFT_MODEL}...')
model, tokenizer = load_fn(SFT_MODEL)
print('Model loaded.')

fresh_pairs = generate_fresh_dpo_pairs(
    model, tokenizer, generate_fn,
    n_prompts=30,     # adjust based on time budget
    n_candidates=4,
)

## 5. Save DPO dataset

In [ ]:
all_dpo_pairs = dpo_pairs + fresh_pairs
random.shuffle(all_dpo_pairs)

# Split 90/10 train/valid
split_idx = int(len(all_dpo_pairs) * 0.9)
train_pairs = all_dpo_pairs[:split_idx]
valid_pairs = all_dpo_pairs[split_idx:]

# Save in the format mlx-lm DPO expects
for name, pairs in [('train', train_pairs), ('valid', valid_pairs)]:
    out_path = DPO_DATA_DIR / f'{name}.jsonl'
    with open(out_path, 'w') as f:
        for pair in pairs:
            f.write(json.dumps(pair) + '\n')
    print(f'Saved {len(pairs)} DPO pairs to {out_path}')

print(f'\nTotal DPO pairs: {len(all_dpo_pairs)} (train={len(train_pairs)}, valid={len(valid_pairs)})')

## 6. Run DPO training

In [ ]:
# DPO training via mlx-lm
# mlx-lm supports DPO via the `lora` command with --dpo flag

BATCH_SIZE    = 1
GRAD_ACCUM    = 8
LORA_LAYERS   = 8
LORA_RANK     = 16
LEARNING_RATE = 5e-6    # lower than SFT — DPO is sensitive to LR
DPO_ITERS     = 200     # DPO converges faster than SFT
DPO_BETA      = 0.1     # KL penalty coefficient

if len(all_dpo_pairs) < 10:
    print('⚠  Too few DPO pairs (<10). Skipping DPO training.')
    print('   Run the full pipeline first to accumulate rejects, then re-run this notebook.')
else:
    train_cmd = [
        sys.executable, '-m', 'mlx_lm', 'lora',
        '--model',                   SFT_MODEL,
        '--data',                    str(DPO_DATA_DIR),
        '--train',
        '--batch-size',              str(BATCH_SIZE),
        '--grad-accumulation-steps', str(GRAD_ACCUM),
        '--num-layers',              str(LORA_LAYERS),
        '--learning-rate',           str(LEARNING_RATE),
        '--iters',                   str(DPO_ITERS),
        '--save-every',              '50',
        '--val-batches',             '10',
        '--adapter-path',            str(DPO_ADAPTER),
        '--max-seq-length',          '4096',
    ]

    print('Starting DPO training...')
    print(' '.join(train_cmd))
    print()

    proc = subprocess.Popen(
        train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line.rstrip())
    proc.wait()

    if proc.returncode != 0:
        print(f'\n⚠  DPO training exited with code {proc.returncode}')
    else:
        print(f'\n✓  DPO training complete. Adapter saved to: {DPO_ADAPTER}')

## 7. Fuse DPO adapter

In [ ]:
if (DPO_ADAPTER / 'adapters.safetensors').exists():
    fuse_cmd = [
        sys.executable, '-m', 'mlx_lm.fuse',
        '--model',        SFT_MODEL,
        '--adapter-path', str(DPO_ADAPTER),
        '--save-path',    str(DPO_FUSED),
    ]

    print('Fusing DPO adapter...')
    print(' '.join(fuse_cmd))
    result = subprocess.run(fuse_cmd)

    if result.returncode != 0:
        raise RuntimeError(f'Fuse failed with exit code {result.returncode}')

    print(f'✓  DPO fused model saved to: {DPO_FUSED}')
else:
    print('⚠  No DPO adapter found — skipping fuse.')

## 8. Smoke test

In [ ]:
if DPO_FUSED.exists() and (DPO_FUSED / 'config.json').exists():
    print(f'Loading DPO model from {DPO_FUSED}...')
    dpo_model, dpo_tokenizer = load_fn(str(DPO_FUSED))

    kb = load_knowledge()
    sys_prompt = build_system_prompt(kb)

    test_messages = [
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': 'Write an ARO feature set that retrieves a user by ID and returns an OK response.'},
    ]
    prompt = dpo_tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    response = generate_fn(dpo_model, dpo_tokenizer, prompt=prompt, max_tokens=300, verbose=False)

    print('\n=== DPO Smoke Test ===')
    print(response)

    # Quick syntax check
    code = extract_aro_code(response)
    if aro_check(code):
        print('\n✓  Syntax check passed')
    else:
        print('\n⚠  Syntax check failed')
else:
    print('No DPO model available for smoke test.')

## Summary

In [ ]:
print('=' * 60)
print('notebook 17 — DPO PREFERENCE TRAINING SUMMARY')
print('=' * 60)

print(f'\nData:')
print(f'  Rejects loaded:       {len(all_rejects)}')
print(f'  Close-but-wrong:      {len(filtered_rejects)}')
print(f'  DPO pairs from rejects: {len(dpo_pairs)}')
print(f'  Fresh DPO pairs:      {len(fresh_pairs)}')
print(f'  Total DPO pairs:      {len(all_dpo_pairs)}')

print(f'\nModel:')
print(f'  SFT base:  {SFT_MODEL}')
if DPO_FUSED.exists():
    size_gb = sum(f.stat().st_size for f in DPO_FUSED.rglob('*') if f.is_file()) / 1e9
    print(f'  DPO fused: {DPO_FUSED} ({size_gb:.1f} GB)')
else:
    print(f'  DPO fused: not yet created')

print(f'\nNext steps:')
print(f'  1. Run NB18 (evaluation) to compare SFT vs DPO model quality')
print(f'  2. Run NB20 (packaging) — it will pick up the DPO fused model automatically')
print('=' * 60)